# IGFL confocal MERFISH pipeline — beginner-friendly run

**Goal:** run the full `merfish3d-analysis` pipeline on **igfl Abberior confocal**
MERFISH data, one step at a time, **without** memorizing any command-line syntax.
Edit a single path, then run the cells top to bottom.

Each step below is a thin wrapper around a console command. Every option is written
out explicitly at its default value so you can see — and tweak — exactly what runs.

> ⚠️ **This notebook runs locally and needs a GPU.** It is *not* a Google Colab
> notebook. You must launch it from a workstation that already has `merfish3d-analysis`
> installed. If nothing is installed yet, follow the **Quick start for non-coders**
> section in the project `README.md` first.

## Prerequisites (one-time, done by whoever set up this machine)

- Linux + an Nvidia GPU capable of CUDA 12.8
- The **`merfish3d`** conda environment, installed with `setup-merfish3d`
- The **`merfish3d-stitcher`** conda environment — created automatically by
  `setup-merfish3d`. This second environment exists only for the global-registration
  step (a numpy-version conflict workaround) and **this notebook switches to it
  automatically** when needed (see Step 3).
- This notebook's kernel must be the **`merfish3d`** environment.


## Configure your data

Set `ROOT_PATH` to the folder that holds your acquisition. That folder **must**
contain:

```
<ROOT_PATH>/
├── Raw data/        # raw OME-TIFF stacks from the Abberior confocal
├── Deconv data/     # Huygens-deconvolved OME-TIFF stacks
└── codebook.csv     # semicolon-delimited codebook
```

You only edit the path in **this one cell** — every step below reuses it.


In [1]:
from pathlib import Path

# 👇 EDIT THIS: the folder containing "Raw data/", "Deconv data/", and codebook.csv
ROOT_PATH = Path("/home/hblanc01/Data/20250718_DH_Merfish_3x2bit")

# The pipeline always reads and writes the datastore at <ROOT_PATH>/qi2labdatastore.
# (All steps after the first use this fixed location, so you do not choose it.)
datastore_path = ROOT_PATH / "qi2labdatastore"
state_json = datastore_path / "datastore_state.json"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"datastore_path = {datastore_path}")

ROOT_PATH      = /home/hblanc01/Data/20250718_DH_Merfish_3x2bit
datastore_path = /home/hblanc01/Data/20250718_DH_Merfish_3x2bit/qi2labdatastore


### Helper functions (run once)

`run()` executes a console command and shows its output live. `already_done()`
peeks at the datastore's small `datastore_state.json` sidecar so a step can
**skip itself** if it has already completed — this protects hours of finished GPU
work from being overwritten if you re-run the notebook. Pass `force=True` to a step
to run it anyway.


In [2]:
import json
import shlex
import subprocess


def run(cmd: str, env: str | None = None) -> None:
    """Run a shell command, streaming its output into the notebook.

    Parameters
    ----------
    cmd : str
        The command line to execute.
    env : str or None
        If given, run the command inside that conda environment via
        ``conda run -n <env> ...``. Used for the global-registration step.
    """
    if env is not None:
        cmd = f"conda run --no-capture-output -n {env} {cmd}"
    print(f"$ {cmd}\n")
    proc = subprocess.run(shlex.split(cmd))
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")


def already_done(*flags: str) -> bool:
    """Return True if every named flag is already True in datastore_state.json."""
    if not state_json.exists():
        return False
    with open(state_json) as fh:
        state = json.load(fh)
    return all(bool(state.get(flag, False)) for flag in flags)

## Step 0 — Sanity check (no GPU)

Confirm the input folder looks right before spending any GPU time. This raises a
clear error if something is missing.


In [3]:
if not ROOT_PATH.exists():
    raise FileNotFoundError(f"ROOT_PATH does not exist: {ROOT_PATH}")

required = ["Raw data", "Deconv data", "codebook.csv"]
missing = [name for name in required if not (ROOT_PATH / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required item(s) in {ROOT_PATH}: {missing}. "
        "Expected 'Raw data/', 'Deconv data/', and 'codebook.csv'."
    )

print("✓ Input folder looks good — ready to run the pipeline.")

✓ Input folder looks good — ready to run the pipeline.


## Step 1 — Build the datastore  (`igfl-datastore`)

Reads the raw + deconvolved OME-TIFFs and packs them into a single zarr-backed
**qi2labDataStore**. Pixel size, NA, and channels are parsed from the OME metadata.
This is a one-time conversion; later steps all read and write through this datastore.

*Runs in the `merfish3d` env. Typically a few minutes depending on data size.*


In [14]:
force = False  # set True to rebuild even if the datastore already exists

if datastore_path.exists() and not force:
    print(f"✓ Datastore already exists at {datastore_path} — skipping.")
    print("  (set force=True above to rebuild from scratch)")
else:
    run(
        f'igfl-datastore "{ROOT_PATH}"'
        " --deconv-scale 200.0"  # multiplier applied to Huygens float before uint16 cast
        # first Z plane of the crop window (with --z-size-crop)
        " --z-start 120"
        " --z-size-crop 30"  # crop Z to N planes (memory/test), need >25
        # " --ri 1.4"            # refractive index of immersion medium (silicon oil)
        # " --ri-sample 1.33"    # refractive index of sample
        # "--na FLOAT"          # objective NA; omitted -> auto-parsed from OME metadata
    )

$ igfl-datastore "/home/hblanc01/Data/20250718_DH_Merfish_3x2bit" --deconv-scale 200.0 --z-start 120 --z-size-crop 30

Conversion metadata written to /home/hblanc01/Data/20250718_DH_Merfish_3x2bit/conversion_metadata.json


tile:   0%|          | 0/6 [00:00<?, ?it/s]

bits:   0%|          | 0/3 [00:00<?, ?it/s]

bits:  33%|███▎      | 1/3 [00:00<00:00,  5.76it/s]

bits:  67%|██████▋   | 2/3 [00:00<00:00,  5.76it/s]

bits: 100%|██████████| 3/3 [00:00<00:00,  6.04it/s]

                                                   
tile:  17%|█▋        | 1/6 [02:10<10:51, 130.21s/it]

bits:   0%|          | 0/3 [00:00<?, ?it/s]

bits:  33%|███▎      | 1/3 [00:00<00:00,  3.80it/s]

bits:  67%|██████▋   | 2/3 [00:00<00:00,  5.14it/s]

bits: 100%|██████████| 3/3 [00:00<00:00,  5.98it/s]

                                                   
tile:  33%|███▎      | 2/6 [04:06<08:08, 122.15s/it]

bits:   0%|          | 0/3 [00:00<?, ?it/s]

bits:  33%|███▎      | 1/3 [00:00<00:01,  1.99it/s]

bits:  67%|██████▋   | 2/3 [00:01<00:00,  1.86it/s]

bits: 100%|██████████| 3/3 [00:01<00:00,  2.73it/s]

                                                   
tile:  50%|█████     | 3/6 [05:51<05:42, 114.26s/it]

bits:   0%|          | 0/

Done. Datastore written to /home/hblanc01/Data/20250718_DH_Merfish_3x2bit/qi2labdatastore


## Step 2 — Preprocess & local registration  (`qi2lab-preprocess`)

Registers every imaging round to the first round **within each tile** (local
coordinates). Optical-flow refinement and U-FISH feature detection improve the
alignment.

> ℹ️ **Deconvolution is turned off here (`--no-decon --no-decon-allfiducial`).**
> This igfl confocal data is already deconvolved by Huygens (that is what the
> `Deconv data/` folder is). Running deconvolution again would degrade the signal,
> so we skip it. For raw, *non*-deconvolved data you would instead leave `--decon` on.

*Runs in the `merfish3d` env. GPU-heavy — can take a while on large data.*


In [17]:
force = True

if already_done("Corrected", "LocalRegistered") and not force:
    print("✓ Already preprocessed and locally registered — skipping.")
    print("  (set force=True above to re-run; note it overwrites prior results)")
else:
    run(
        f'qi2lab-preprocess "{ROOT_PATH}"'
        # " --num-gpus 1"             # number of GPUs to use
        " --no-decon"               # data is already Huygens-deconvolved -> don't redo it
        " --no-decon-allfiducial"   # likewise skip deconvolution of the fiducial rounds
        # optical-flow registration (--no-opticalflow to skip)
        " --opticalflow"
        " --save-all-fiducial"
        # " --crop-yx-decon 2048"     # tile size used only if deconvolution is enabled
        # --ufish-model TEXT        # U-FISH model; omitted -> package default (simfish)
    )

$ qi2lab-preprocess "/home/hblanc01/Data/20250718_DH_Merfish_3x2bit" --no-decon --no-decon-allfiducial --opticalflow --save-all-fiducial

Using datastore at /home/hblanc01/Data/20250718_DH_Merfish_3x2bit/qi2labdatastore
2026-06-12 17:31:28 Finished fiducial tile id: tile0000; round id: round001.
2026-06-12 17:31:39 Finished fiducial tile id: tile0000; round id: round002.
2026-06-12 17:31:51 Finished readout tile id: tile0000; bit id: bit001.
2026-06-12 17:32:00 Finished readout tile id: tile0000; bit id: bit002.
2026-06-12 17:32:07 Finished readout tile id: tile0000; bit id: bit003.
2026-06-12 17:32:15 Finished readout tile id: tile0000; bit id: bit004.
2026-06-12 17:32:24 Finished readout tile id: tile0000; bit id: bit005.
2026-06-12 17:32:31 Finished readout tile id: tile0000; bit id: bit006.
2026-06-12 17:32:33 Finished fiducial tile id: tile0001; round id: round001.
2026-06-12 17:32:40 Finished fiducial tile id: tile0001; round id: round002.
2026-06-12 17:32:52 Finished readout til

## Step 3 — Global registration  (`qi2lab-globalregister`)  🔀 *env switch*

Places all tiles into one global coordinate system and fuses them (multiview
stitching).

> 🔀 **This step runs in a different conda environment.** Global registration uses
> `multiview-stitcher`, which lives in the separate **`merfish3d-stitcher`**
> environment (a numpy-version conflict means it can't share the `merfish3d` env).
> A single notebook kernel can't change its own environment, so this cell shells out
> with `conda run -n merfish3d-stitcher ...`. You don't need to do anything — just run
> the cell. Every *other* step runs in the normal `merfish3d` env.

*GPU- and memory-heavy.*


In [ ]:
force = False

if already_done("GlobalRegistered", "Fused") and not force:
    print("✓ Already globally registered and fused — skipping.")
    print("  (set force=True above to re-run)")
else:
    run(
        f'qi2lab-globalregister "{ROOT_PATH}"'
        " --n-jobs 16"            # parallel fusion jobs
        " --fused-chunk-size 128"  # fused image chunk size
        # " --create-max-proj-tiff"  # write max-projection TIFF for cellpose tuning
        # --swap-yx              # add this flag if stage X/Y are swapped
        ,
        env="merfish3d-stitcher",  # 🔀 the env switch happens here
    )

$ conda run --no-capture-output -n merfish3d-stitcher qi2lab-globalregister "/home/hblanc01/Data/20250718_DH_Merfish_3x2bit" --n-jobs 16 --fused-chunk-size 128 --create-max-proj-tiff



Traceback (most recent call last):
  File "/home/hblanc01/miniforge3/envs/merfish3d-stitcher/bin/qi2lab-globalregister", line 3, in <module>
    from merfish3danalysis.cli.qi2lab_microscopes.global_register import main
  File "/home/hblanc01/Github/merfish3d-analysis/src/merfish3danalysis/cli/qi2lab_microscopes/global_register.py", line 16, in <module>
    import matplotlib.pyplot as plt
  File "/home/hblanc01/miniforge3/envs/merfish3d-stitcher/lib/python3.12/site-packages/matplotlib/__init__.py", line 1299, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
    ~~~~~~~~^^^^^^^^^^^
  File "/home/hblanc01/miniforge3/envs/merfish3d-stitcher/lib/python3.12/site-packages/matplotlib/__init__.py", line 774, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qt

RuntimeError: Command failed (exit 1): conda run --no-capture-output -n merfish3d-stitcher qi2lab-globalregister "/home/hblanc01/Data/20250718_DH_Merfish_3x2bit" --n-jobs 16 --fused-chunk-size 128 --create-max-proj-tiff

## Step 4 — Segment cells  (`igfl-segment`)

Runs 3D Cellpose on the fused fiducial volume to find cell boundaries and saves
ROIs in global coordinates.

The defaults below are a starting point — `--diameter` (expected cell size in
pixels) is the knob you'll most often adjust for your sample.

*Runs in the `merfish3d` env.*


In [ ]:
force = False

if already_done("SegmentedCells") and not force:
    print("✓ Cells already segmented — skipping.")
    print("  (set force=True above to re-run, e.g. after changing --diameter)")
else:
    run(
        f'igfl-segment "{ROOT_PATH}"'
        " --diameter 30"            # expected cell diameter in pixels
        " --flow-threshold 0.4"     # cellpose flow error threshold
        " --cellprob-threshold 0.0"  # cell probability threshold
        " --normalization 1.0 99.0"  # percentile normalization [low high]
    )

## Step 5 — Decode transcripts  (`qi2lab-decode`)

The payoff step: GPU pixel decoding turns the registered bits into **decoded RNA
molecules** (gene identity + 3D position). Thresholds are auto-selected for your
data by default; the options below are shown so you can override them if needed.

*Runs in the `merfish3d` env. GPU-heavy.* Output lands at
`<datastore>/all_tiles_filtered_decoded_features/decoded_features.csv.gz`.


In [ ]:
force = False

if already_done("DecodedSpots", "FilteredSpots") and not force:
    print("✓ Transcripts already decoded and filtered — skipping.")
    print("  (set force=True above to re-run)")
else:
    run(
        f'qi2lab-decode "{ROOT_PATH}"'
        " --num-gpus 1"                  # number of GPUs to use
        " --filter-method blank_fraction"  # transcript filter method
        " --normalization-method iterative"  # normalization source for decoding
        # --minimum-pixels-per-rna INT   # omitted -> auto (28 for 3D, 7 for 2D)
        # --feature-predictor-threshold FLOAT  # omitted -> auto by axial sampling
        # --magnitude-threshold FLOAT FLOAT    # omitted -> auto by axial sampling
        # --skip-optimization            # add to skip iterative self-optimization
    )

## Step 6 — View your results

The decoded transcripts are a gzipped CSV. The cell below previews the first rows.
For an interactive 3D view of tiles, fiducials, segmentation, and decoded molecules,
launch the viewer in a **terminal** (not here):

```bash
qi2lab-viewer "<ROOT_PATH>"
```


In [ ]:
import pandas as pd

decoded_csv = (
    datastore_path
    / "all_tiles_filtered_decoded_features"
    / "decoded_features.csv.gz"
)

if decoded_csv.exists():
    df = pd.read_csv(decoded_csv)
    print(f"{len(df):,} decoded transcripts at {decoded_csv}\n")
    display(df.head())
else:
    print(f"No decoded file found yet at {decoded_csv}.")
    print("Run Step 5 first.")